In [ ]:
import os
import json
import os
from glob import glob
from typing import Optional, List, Dict, Set
import pandas as pd
from collections import defaultdict
import numpy as np
import re
from tqdm import tqdm



In [ ]:
agents = ["codex","gemini","claude","gpt-5","gemini-2.5-pro","sonnet-4.5"]
# agents = ["codex"]

graders = ["gpt-5","gemini-2.5-pro","sonnet"]

### Load grade json

In [ ]:
def find_grade_files(results_dir: str, agent: str, grader: Optional[str] = None) -> List[str]:
    agent_dir = os.path.join(results_dir, agent)
    if not os.path.isdir(agent_dir):
        raise FileNotFoundError(f"Agent results directory not found: {agent_dir}")

    pattern = os.path.join(agent_dir, "task-*", "grades", grader if grader else "*", "trial-*-grade.json")
    files = sorted(glob(pattern))
    return files

In [ ]:

def load_grades(files: List[str]) -> List[Dict]:
    aggregated = []
    for path in files:
        try:
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception as e:
            # Skip unreadable files but note the error in metadata
            aggregated.append({
                "path": path,
                "error": str(e),
            })
            continue

        # Extract simple metadata from path: task id and grader
        # Expected path: .../<agent>/task-XX/grades/<grader>/task-XX-grade.json
        parts = path.replace("\\", "/").split("/")
        task_id = None
        grader = None
        try:
            # Find indices for known segments
            if "grades" in parts:
                gi = parts.index("grades")
                # parts[gi-1] should be task-XX, parts[gi+1] is grader
                task_id = parts[gi - 1] if gi - 1 >= 0 else None
                grader = parts[gi + 1] if gi + 1 < len(parts) else None
        except Exception:
            pass

        aggregated.append({
            "path": path,
            "task": task_id,
            "grader": grader,
            "data": data,
        })
    return aggregated

In [ ]:
def load_criteria_classes(evaluations_dir: str) -> Dict[str, str]:
    path = os.path.join(evaluations_dir,"results", "criteria_classes.json")
    if not os.path.isfile(path):
        raise FileNotFoundError("criteria_classes.json not found at {}".format(path))
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, dict):
        raise ValueError("criteria_classes.json must be a JSON object mapping criterion -> class")
    # Normalize keys to allow simple matching; keep original form for output
    return {k: v for k, v in data.items()}

In [ ]:

def extract_task_and_grader(path: str) -> Dict[str, Optional[str]]:
    parts = path.replace("\\", "/").split("/")
    task_id = None
    grader = None
    if "grades" in parts:
        gi = parts.index("grades")
        task_id = parts[gi - 1] if gi - 1 >= 0 else None
        grader = parts[gi + 1] if gi + 1 < len(parts) else None
    return {"task": task_id, "grader": grader}

In [ ]:
def get_trial_number(path):
    match = re.search(r'trial-(\d+)-grade\.json', path)

    return int(match.group(1))

def aggregate_file(path: str, classes_map: Dict[str, str]) -> Dict:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    stats = data.get("stats") if isinstance(data, dict) else {}
    trial_overalls = []
    if isinstance(stats, dict):
        to = stats.get("trial_overalls")
        if isinstance(to, list):
            trial_overalls = [float(x) for x in to if isinstance(x, (int, float))]
    


    criteria_stats = stats.get("criteria") if isinstance(stats, dict) else None
    criteria_info = []
    if isinstance(criteria_stats, list):
        for entry in criteria_stats:
            if isinstance(entry, dict) and isinstance(entry.get("criterion"), str):
                crit = entry["criterion"].strip()
                avg = entry["average"]
                std = entry["stddev"]
                mapped_class = classes_map.get(crit)
                criteria_info.append({
                    "criterion": crit,
                    "average":avg,
                    "stddev":std,
                    "class": mapped_class if mapped_class is not None else "Unknown"
                })

    meta = extract_task_and_grader(path)
    return {
        "path": path,
        "task": meta.get("task"),
        'trial': get_trial_number(path),
        "grader": meta.get("grader"),
        "trial_overalls": trial_overalls,
        "criteria_info": criteria_info,
    }

In [ ]:

def build_nested_structure() -> Dict:
    all_data = {}

    for agent in agents:
        nested = defaultdict(lambda: defaultdict(list))  # task → grader → list of trials

        for grader in graders:
            paths = find_grade_files("../run-and-grade/results/",agent, grader)
            classes_map = load_criteria_classes(".")
            for path in paths:
                trial_data = aggregate_file(path, classes_map)                
    
                task = trial_data.pop("task")
                # grader = trial_data.pop("grader")
                trial_number = trial_data.pop("trial")

                trial_entry = {
                    "trial": trial_number,
                    "path": trial_data["path"],
                    "trial_overalls": trial_data["trial_overalls"],
                    "criteria_info": trial_data["criteria_info"]
                }

                nested[task][grader].append(trial_entry)
        all_data[agent] = nested

    return all_data


##### Read all grade files 

In [ ]:
all_data = build_nested_structure()

In [ ]:
all_data['codex']

### Aggregating all trials 

In [ ]:
all_tasks = list(all_data['codex'].keys())
all_criteria = []

for task in all_tasks:
    task_criteria = all_data['codex'][task]['gpt-5'][0]["criteria_info"]
    for crit in task_criteria:
        all_criteria.append(crit["criterion"])




In [ ]:
all_criteria = list(set(all_criteria))
all_criteria = [crit for crit in all_criteria if crit not in ['TOTAL SCORE (Average)',"TOTAL SCORE**** (****Average****)"]]

In [ ]:
aggregated_info = {}
for agent in agents:
    all_task  = {}

    all_tasks = list(all_data[agent].keys())
    for task in all_tasks:
        all_grader = {}

        for grader in graders:
            trials = all_data[agent][task][grader]
            all_trail_mean = []
            for trial in trials:
                trial_overall = trial["trial_overalls"]
                mean_trial_overall = np.mean(trial_overall)
                all_trail_mean.append(mean_trial_overall)
            
            all_mean = np.mean(all_trail_mean)
            all_std = np.std(all_trail_mean)

            all_grader[grader] = {"mean":all_mean, "std":all_std}
        
        all_task[task] = all_grader
    
    aggregated_info[agent] = all_task
                   

        

In [ ]:
len(aggregated_info["gemini"])

## Aggregating trials by criteria class

In [ ]:
all_data['codex']['task-100']['gpt-5'][0]["criteria_info"]

In [ ]:
len(all_data['codex']['task-100']['gpt-5'])

In [ ]:
all_grader_task_criteria_score = {}

for agent in agents:
    grader_score = {}
    for grader in graders:
        task_criteria_score = {}
        for task in all_tasks:

            all_trials = all_data[agent][task][grader]
            score_by_trial = defaultdict(list)
            for trial in all_trials:
                criteria = trial["criteria_info"]

                for crit in criteria:
                    
                    criteria_class = crit["class"]
                    criteria_score = crit["average"]
                    if criteria_class != "Unknown":
                        score_by_trial[criteria_class].append(criteria_score)
            
            aggregate_score = {}
            for criteria, crit_score in score_by_trial.items():
                aggregate_score[criteria] = np.mean(crit_score)

            task_criteria_score[task] = aggregate_score
        
        grader_score[grader] = task_criteria_score.copy()
        
    all_grader_task_criteria_score[agent] = grader_score
    

        


In [ ]:
all_grader_task_criteria_score['codex']['gpt-5']

In [ ]:
def get_aggregated_trial_result(data):
    models = ['gpt-5', 'gemini-2.5-pro']
    agents = list(data.keys())  # Dynamically get all agent names
    overall = {model: [] for model in models}

    for model in models:
        for agent in agents:
            scores = []
            for task in data.get(agent, {}):
                task_id = int(task.split("-")[1])
                task_scores = data[agent][task].get(model)
                if task_scores:
                    task_mean = float(task_scores["mean"])
                    scores.append(task_mean)
            overall[model].append(scores)
        
    return overall

In [ ]:
overall_result = get_aggregated_trial_result(aggregated_info)
overall_result

### Get report length

In [ ]:
def extract_task_from_path(path):
    # Extract the filename from the path
    # filename = os.path.basename(path)
    
    # Use regex to extract the task portion (e.g., task-1)
    match = re.search(r'(task-\d+)', path)
    
    if match:
        return match.group(1)
    else:
        return None

In [ ]:
def find_report_files(results_dir: str, agent: str) -> List[str]:
    agent_dir = os.path.join(results_dir, agent)
    if not os.path.isdir(agent_dir):
        raise FileNotFoundError(f"Agent results directory not found: {agent_dir}")

    pattern = os.path.join(agent_dir, "task-*","trial-*","out","report.md")
    files = sorted(glob(pattern))
    return files

In [ ]:

def count_words_in_markdown(file_path: str) -> int:
    if not os.path.isfile(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")
    
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    words = content.split()
    return len(words)


In [ ]:
all_agent_word_count = {}
for agent in agents:
    all_word_count = defaultdict(list)
    agent_reports = find_report_files("../run-and-grade/results",agent)
    for report in tqdm(agent_reports):
        task_id = extract_task_from_path(report)
        word_count = count_words_in_markdown(report)
        all_word_count[task_id].append(word_count)
    
    mean_count = {}
    for task_id, word_counts in all_word_count.items():
        mean_count.update({task_id: np.mean(word_counts)})

    
    all_agent_word_count.update({agent:mean_count})
        


### Read metadata 

In [ ]:

def extract_task_and_feature_map(df,
                            task_col='task',
                            feat_col='Difficulty estimate'):
    """
    Returns a dict mapping each task name to its difficulty estimate.
    
    Args:
      df        : pandas.DataFrame containing your data.
      task_col  : column name for task identifiers.
      diff_col  : column name for difficulty estimates.
    
    Returns:
      dict: { task_name: difficulty_value, … }
    """
    # Drop rows where either column is missing
    sub = df[[task_col, feat_col]].dropna()
    
    # Build and return the mapping
    return dict(zip(sub[task_col], sub[feat_col]))


In [ ]:
meta_data = pd.read_json("./metadata.json")
meta_data.head()

### Save final result as csv

In [ ]:
task_diff = extract_task_and_feature_map(df=meta_data,task_col='task',
                            feat_col='difficulty')

task_agency = extract_task_and_feature_map(df=meta_data,task_col='task',
                            feat_col='agency')


task_ref = extract_task_and_feature_map(df=meta_data,task_col='task',
                            feat_col='total_ref')


task_eis_title = extract_task_and_feature_map(df=meta_data,task_col='task',
                            feat_col='eis_title')

task_eis_section = extract_task_and_feature_map(df=meta_data,task_col='task',
                            feat_col='eis_section')

task_ground_truth_len = extract_task_and_feature_map(df=meta_data,task_col='task',
                            feat_col='ground_truth_length')

In [ ]:
all_score_len_data = []
for agent in agents:
    for task, length in all_agent_word_count[agent].items():
        for judge in graders:
            try:
                score = aggregated_info[agent][task][judge]['mean']
                difficulty = task_diff[task]
                agency = task_agency[task]
                ref = task_ref.get(task)
                ground_truth_length = task_ground_truth_len[task]
                eis_title = task_eis_title.get(task)
                eis_section = task_eis_section.get(task)

                accuracy = all_grader_task_criteria_score[agent][judge][task].get("Accuracy")
                clarity = all_grader_task_criteria_score[agent][judge][task].get("Clarity")
                structure = all_grader_task_criteria_score[agent][judge][task].get("Structure")
                reference = all_grader_task_criteria_score[agent][judge][task].get("Reference")

                
                all_score_len_data.append({"agent":agent, "task":task,  "length":length,
                                        "ground_truth_length":ground_truth_length,
                                        "difficulty":difficulty, "agency":agency,"judge":judge,
                                        "total_ref":ref, "eis_title":eis_title, "eis_section":eis_section, "score":score, 
                                        "accuracy":accuracy,"clarity":clarity, "structure":structure, 
                                        "reference":reference })
            except:
                print(agent, task, judge)
                raise

In [ ]:
df_all = pd.DataFrame(all_score_len_data)

In [ ]:
df_all.head()

In [ ]:
df_all.to_csv("result.csv", index=False)